In [1]:
import requests
from bs4 import BeautifulSoup
from pathlib import Path
import json
import random
import time
from urllib.parse import urlencode

from selenium import webdriver
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


BASE_URL = "https://nezamat.ir/"

def build_url(page_number):
    if page_number == 1:
        return BASE_URL
    return BASE_URL + "page/" + str(page_number)

In [ ]:
BRAVE_PATH = r"C:\\Program Files\\Waterfox\\waterfox.exe"

def create_driver(headless=False):
    options = Options()
    options.binary_location = BRAVE_PATH
    if headless:
        options.add_argument("--headless=new")

    options.add_argument("--start-maximized")
    options.add_argument("--disable-notifications")
    driver = webdriver.Firefox(options=options)
    return driver

In [ ]:
def scrape_page(driver, page_number, timeout=60, max_retries=5):
    url = build_url(page_number)
    for attempt in range(1, max_retries + 1):
        print(f"\nLoading page {page_number} (attempt {attempt}/{max_retries})")
        print(url)
        try:
            driver.get(url)
            wait = WebDriverWait(driver, timeout)
            wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div.elementor-element[data-element_type='container']")))
            cards = driver.find_elements(By.CSS_SELECTOR, "h2.elementor-heading-title a[href]")
            documents = []
            for element in cards:

                title = element.text.strip()
                href = element.get_attribute("href")
                if not href:
                    continue
                if "nezamat.ir/" not in href:
                    continue
                card = element.find_element(By.XPATH, "./ancestor::div[contains(@class, 'e-parent')][1]")

                excerpt = ""
                try:
                    excerpt = card.find_element( By.CSS_SELECTOR, "div.elementor-element.e-parent:has(h2.elementor-heading-title a[href])").text.strip()
                except Exception:
                    pass
                metadata = card.find_elements(By.CSS_SELECTOR, "span.elementor-heading-title")

                approval_date = ""
                number = ""
                publication_date = ""
                category = ""
                category_url = ""

                for item in metadata:
                    text = item.text.strip()

                    if text.startswith("تصویب:"):
                        approval_date = text.replace("تصویب:", "", 1).strip()
                    elif text.startswith("شماره:"):
                        number = text.replace("شماره:", "", 1).strip()
                    elif text.startswith("انتشار:"):
                        publication_date = text.replace("انتشار:", "", 1).strip()
                    elif text.startswith("دسته:"):
                        category = text.replace("دسته:", "", 1).strip()
                        try:
                            category_url = item.find_element(By.CSS_SELECTOR, "a[href]").get_attribute("href")
                        except Exception:
                            pass

                documents.append({
                    "title": title,
                    "url": href,
                    "excerpt": excerpt,
                    "approval_date": approval_date,
                    "number": number,
                    "publication_date": publication_date,
                    "category": category,
                    "category_url": category_url,
                    "source": "nezamat.ir"
                })
            print(
                f"Found {len(documents)} documents"
            )
            return documents
        except Exception as e:
            print(f"Page {page_number} failed (attempt {attempt}/{max_retries})")
            source = driver.page_source.lower()
            if "runtime error" in source:
                print("Detected ASP.NET Runtime Error.")
            elif "transferring to the website" in source:
                print("Detected AbrArvan transfer page.")
            else:
                print(f"Unexpected error: {type(e).__name__}")
            wait_time = min(3 * attempt, 15)
            print(f"Retrying in {wait_time} seconds...")
            time.sleep(wait_time)
    return []

In [ ]:
def scrape_all(start_page=1, max_pages=None, delay=(2, 2.5), headless=False):
    driver = create_driver(headless=headless)
    all_documents = []
    try:
        page = start_page
        while True:
            if max_pages is not None:
                if page >= start_page + max_pages:
                    break
            documents = scrape_page(driver, page)
            print(f"Found {len(documents)} documents")
            if not documents:
                print("No documents found. Assuming end of pagination.")
                break
            all_documents.extend(documents)
            print(f"Total unique documents: {len(all_documents)}")
            page += 1
            time.sleep(random.uniform(delay[0], delay[1]))
    finally:
        driver.quit()
    return all_documents

In [ ]:
documents = scrape_all(start_page=1, max_pages=4141, headless=False)

print( "TOTAL:", len(documents))

In [ ]:
documents

In [ ]:
output_path = Path("links")
output_path.mkdir(exist_ok=True, parents=True)

(output_path / "nezamat.txt").write_text("\n\n".join("\n".join(f"{key}---{value}" for key, value in document.items()) for document in documents),encoding="utf-8")

22448544